In [33]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

INPUT_FILE = '../Master-Data/categorized_food_data.csv'

df = pd.read_csv(INPUT_FILE, low_memory=False)

os.makedirs('master_data', exist_ok=True)
print('Data loaded')
print(f'   Rows       : {len(df):,}')
print(f'   Categories : {df["Primary_Category"].nunique()}')
print(f'   Columns    : {list(df.columns)}')
df.head(3)

Data loaded
   Rows       : 47,495
   Categories : 16
   Columns    : ['product_name', 'categories_tags', 'ingredients_text', 'nutriscore_score', 'energy_100g', 'fat_100g', 'saturated-fat_100g', 'carbohydrates_100g', 'sugars_100g', 'fiber_100g', 'proteins_100g', 'salt_100g', 'Primary_Category']


,product_name,categories_tags,ingredients_text,nutriscore_score,energy_100g,fat_100g,saturated-fat_100g,carbohydrates_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g,Primary_Category
0,Pinto Bean,en:asian-style-ready-meal,NaN,NaN,9.0,10.2,3.6,22.5,4.9,3.8,17.5,NaN,Other Snacks
1,pasta,"en:beverages-and-beverages-preparations,en:bev...",NaN,5.0,697.1,6.4,1.3,20.0,1.9,0.8,6.7,0.0009,Other Snacks
2,Eirn original curry Sauce,"en:plant-based-foods-and-beverages,en:plant-ba...",Wheat Flour Sugar Vegetable Fat D Curry Powder...,NaN,1724.0,18.0,8.2,57.2,29.6,1.2,5.2,4.7500,Other Snacks


## The nutrient matrix scatter plat

In [34]:
scatter_df = df[
    (df['sugars_100g'].between(0, 60)) &
    (df['proteins_100g'].between(0, 50))
].copy()


scatter_export = scatter_df[[
    'product_name',
    'Primary_Category',
    'sugars_100g',
    'proteins_100g',
    'fat_100g',
    'fiber_100g',
    'energy_100g'
]].copy()

# Rename columns to clean Power BI field names
scatter_export.columns = [
    'Product Name',
    'Category',
    'Sugar (g per 100g)',
    'Protein (g per 100g)',
    'Fat (g per 100g)',
    'Fiber (g per 100g)',
    'Energy (kcal per 100g)'
]

scatter_export.to_csv('../Master-Data/powerbi_scatter_data.csv', index=False)

print(f'Scatter data exported: {len(scatter_export):,} rows')
print(f'File: powerbi_scatter_data.csv')
scatter_export.head(3)

Scatter data exported: 45,642 rows
File: powerbi_scatter_data.csv


,Product Name,Category,Sugar (g per 100g),Protein (g per 100g),Fat (g per 100g),Fiber (g per 100g),Energy (kcal per 100g)
0,Pinto Bean,Other Snacks,4.9,17.5,10.2,3.8,9.0
1,pasta,Other Snacks,1.9,6.7,6.4,0.8,697.1
2,Eirn original curry Sauce,Other Snacks,29.6,5.2,18.0,1.2,1724.0


## Category summary Table

In [35]:
cat_summary = df.groupby('Primary_Category').agg(
    Median_Sugar   = ('sugars_100g',   'median'),
    Median_Protein = ('proteins_100g', 'median'),
    Median_Fat     = ('fat_100g',      'median'),
    Median_Fiber   = ('fiber_100g',    'median'),
    Product_Count  = ('product_name',  'count')
).round(1).reset_index()

# Clean column names for Power BI
cat_summary.columns = [
    'Category',
    'Median Sugar (g)',
    'Median Protein (g)',
    'Median Fat (g)',
    'Median Fiber (g)',
    'Product Count'
]

cat_summary = cat_summary.sort_values('Median Sugar (g)', ascending=False)

cat_summary.to_csv('../Master-Data/powerbi_category_summary.csv', index=False)

print('Category summary exported')
print('  File: powerbi_category_summary.csv')
print()
print(cat_summary.to_string(index=False))

Category summary exported
  File: powerbi_category_summary.csv

                   Category  Median Sugar (g)  Median Protein (g)  Median Fat (g)  Median Fiber (g)  Product Count
       Sweet Spreads & Dips              60.8                 0.0             0.0               0.0            655
          Chocolate & Candy              46.0                 6.2            27.8               2.6           1330
               Fruit Snacks              45.5                 1.4             0.0               5.0             14
         Biscuits & Cookies              29.6                 5.3            18.8               2.2           1776
    Granola, Bars & Cereals              22.2                 8.3             5.2               7.0           2288
  Ice Cream & Frozen Snacks              18.9                 3.0             8.5               0.6            671
      Yogurt & Dairy Snacks               9.9                 4.6             1.2               0.0           3924
Crackers & Savou

## Gap index Table

In [41]:
PROTEIN_THRESHOLD = 15
SUGAR_THRESHOLD   = 10

gap_data = []

for cat in df['Primary_Category'].unique():
    cat_df       = df[df['Primary_Category'] == cat]
    total        = len(cat_df)
    in_healthy   = len(cat_df[
        (cat_df['proteins_100g'] >= PROTEIN_THRESHOLD) &
        (cat_df['sugars_100g']   <= SUGAR_THRESHOLD)
    ])
    pct_healthy  = round(in_healthy / total * 100, 1)
    pct_gap      = round(100 - pct_healthy, 1)   # gap = how much is NOT healthy

    gap_data.append({
        'Category'                    : cat,
        'Total Products'              : total,
        'Products in Healthy Zone'    : in_healthy,
        '% in Healthy Zone'           : pct_healthy,
        '% Gap (Opportunity Size)'    : pct_gap
    })

gap_df = pd.DataFrame(gap_data).sort_values('% in Healthy Zone', ascending=True)
gap_df.to_csv('../Master-Data/powerbi_gap_index.csv', index=False)

print('Gap index exported')
print('   File: powerbi_gap_index.csv')
print()
print('=== OPPORTUNITY RANKING (lowest % = biggest gap) ===')
for _, row in gap_df.iterrows():
    flag = '  <-- BIGGEST OPPORTUNITY' if row['% in Healthy Zone'] == gap_df['% in Healthy Zone'].min() else ''
    print(f"{row['Category']:<35} {row['% in Healthy Zone']:>5.4f}% in healthy zone{flag}")

Gap index exported
   File: powerbi_gap_index.csv

=== OPPORTUNITY RANKING (lowest % = biggest gap) ===
Ice Cream & Frozen Snacks           0.0000% in healthy zone  <-- BIGGEST OPPORTUNITY
Fruit Snacks                        0.0000% in healthy zone  <-- BIGGEST OPPORTUNITY
Popcorn                             0.0000% in healthy zone  <-- BIGGEST OPPORTUNITY
Biscuits & Cookies                  1.1000% in healthy zone
Yogurt & Dairy Snacks               1.5000% in healthy zone
Crackers & Savoury Biscuits         1.8000% in healthy zone
Chips & Crisps                      1.9000% in healthy zone
Chocolate & Candy                   2.0000% in healthy zone
Sweet Spreads & Dips                2.6000% in healthy zone
Granola, Bars & Cereals             2.9000% in healthy zone
Cakes & Pastries                    9.5000% in healthy zone
Other Snacks                        12.7000% in healthy zone
Nuts, Seeds & Dried Fruit           32.6000% in healthy zone
Cheese Snacks                       50.

## KPI Summary

In [42]:
total_products  = len(df)
in_healthy_zone = len(df[
    (df['proteins_100g'] >= PROTEIN_THRESHOLD) &
    (df['sugars_100g']   <= SUGAR_THRESHOLD)
])
in_danger_zone  = len(df[
    (df['sugars_100g']   > 20) &
    (df['proteins_100g'] < 10)
])

#  Get top 3 opportunity categories
top3 = gap_df.head(3)
opp_cat_1 = top3.iloc[0]['Category']
opp_cat_2 = top3.iloc[1]['Category']
opp_cat_3 = top3.iloc[2]['Category']
opp_pct_1 = top3.iloc[0]['% in Healthy Zone']
opp_pct_2 = top3.iloc[1]['% in Healthy Zone']
opp_pct_3 = top3.iloc[2]['% in Healthy Zone']

kpi_df = pd.DataFrame([{
    'Total Products'            : total_products,
    'Products in Healthy Zone'  : in_healthy_zone,
    '% in Healthy Zone'         : round(in_healthy_zone / total_products * 100, 1),
    'Products in Danger Zone'   : in_danger_zone,
    '% in Danger Zone'          : round(in_danger_zone / total_products * 100, 1),
    'Opportunity Category 1'    : opp_cat_1,
    'Opportunity Gap 1 %'       : opp_pct_1,
    'Opportunity Category 2'    : opp_cat_2,
    'Opportunity Gap 2 %'       : opp_pct_2,
    'Opportunity Category 3'    : opp_cat_3,
    'Opportunity Gap 3 %'       : opp_pct_3,
    'All Opportunities'         : f"{opp_cat_1} | {opp_cat_2} | {opp_cat_3}"
}])

kpi_df.to_csv('../Master-Data/powerbi_kpis.csv', index=False)

print(' KPI summary exported')
print('   File: ../Master-Data/powerbi_kpis.csv')
print()
print(f'   Total products         : {total_products:,}')
print(f'   In Healthy Zone        : {in_healthy_zone:,} ({in_healthy_zone/total_products*100:.1f}%)')
print(f'   In Danger Zone         : {in_danger_zone:,} ({in_danger_zone/total_products*100:.1f}%)')
print()
print('   TOP 3 OPPORTUNITY CATEGORIES:')
print(f'   1. {opp_cat_1:<35} ({opp_pct_1:.1f}% in healthy zone)')
print(f'   2. {opp_cat_2:<35} ({opp_pct_2:.1f}% in healthy zone)')
print(f'   3. {opp_cat_3:<35} ({opp_pct_3:.1f}% in healthy zone)')

 KPI summary exported
   File: ../Master-Data/powerbi_kpis.csv

   Total products         : 47,495
   In Healthy Zone        : 5,420 (11.4%)
   In Danger Zone         : 8,355 (17.6%)

   TOP 3 OPPORTUNITY CATEGORIES:
   1. Ice Cream & Frozen Snacks           (0.0% in healthy zone)
   2. Fruit Snacks                        (0.0% in healthy zone)
   3. Popcorn                             (0.0% in healthy zone)


## Nutri Score Table

In [38]:
# Convert numeric Nutri-Score to letter grade
ns_col = 'nutriscore_score'

# Check if column exists in this dataset
if ns_col not in df.columns:
    print(f"Column '{ns_col}' not found in categorized_food_data.csv")
    print(f"   Available columns: {list(df.columns)}")
    print()
    print("   SOLUTION: The nutri-score column was not carried forward from Sprint 1.")
    print("   Loading it now directly from clean_food_data.csv...")

    # Load nutri-score from Sprint 1 output and merge
    clean_df = pd.read_csv('../Master-Data/clean_food_data.csv',
                           usecols=['product_name', 'sugars_100g',
                                    'proteins_100g', ns_col],
                           low_memory=False)

    df = df.merge(
        clean_df[['product_name', 'sugars_100g', 'proteins_100g', ns_col]],
        on=['product_name', 'sugars_100g', 'proteins_100g'],
        how='left'
    )
    print(f"   Merged! Nutri-score coverage: {df[ns_col].notna().sum():,} products")

ns_df = df.dropna(subset=[ns_col]).copy()

def score_to_grade(score):
    if score <= -1:    return 'A'
    elif score <= 2:   return 'B'
    elif score <= 10:  return 'C'
    elif score <= 18:  return 'D'
    else:              return 'E'

ns_df['Nutri Grade'] = ns_df[ns_col].apply(score_to_grade)

print(f'Products with Nutri-Score data: {len(ns_df):,}')
print(ns_df['Nutri Grade'].value_counts().sort_index().to_string())

Products with Nutri-Score data: 42,673
Nutri Grade
A     5434
B     6524
C    11843
D     9965
E     8907


## Protein Sources

In [39]:
from collections import Counter

# High-protein products only
hp_df = df[
    (df['proteins_100g']    >= PROTEIN_THRESHOLD) &
    (df['ingredients_text'].notna())
].copy()

PROTEIN_SOURCES = [
    'whey', 'peanut', 'soy', 'soya', 'almond', 'chickpea',
    'lentil', 'egg', 'milk protein', 'casein', 'hemp',
    'pea protein', 'rice protein', 'quinoa', 'oat', 'chia', 'sesame'
]

source_counts = Counter()
for text in hp_df['ingredients_text'].str.lower():
    for source in PROTEIN_SOURCES:
        if source in str(text):
            source_counts[source] += 1

protein_sources_df = pd.DataFrame(
    source_counts.most_common(10),
    columns=['Protein Source', 'Product Count']
)
protein_sources_df['Rank'] = range(1, len(protein_sources_df) + 1)

protein_sources_df.to_csv('../Master-Data/powerbi_protein_sources.csv', index=False)

print('   Protein sources exported')
print('   File: powerbi_protein_sources.csv')
print()
print('Top 3 protein sources:')
for _, row in protein_sources_df.head(3).iterrows():
    print(f"  #{int(row['Rank'])} {row['Protein Source'].title():<20} {int(row['Product Count']):,} products")

   Protein sources exported
   File: powerbi_protein_sources.csv

Top 3 protein sources:
  #1 Soy                  815 products
  #2 Peanut               468 products
  #3 Oat                  302 products
